# FDR-Corrected Cache Decay Verdict\n\nThis notebook demonstrates the **FDR-corrected re-analysis of a per-key cache-decay admission policy vs. a global-reset TinyLFU baseline**, for read-heavy key-value stores with skewed (Zipf) key popularity that drifts over time (hot keys change identity).\n\nThe original evaluation (`eval.py`) re-analyzes results from a 108-cell simulation sweep (`method.py`) without re-running it, and:\n\n1. Applies **Benjamini-Hochberg / Benjamini-Yekutieli FDR correction** to 36 per-group bootstrap significance tests on \"does the proposed estimator recover from popularity drift faster?\"\n2. Re-simulates a **CoV-threshold sensitivity grid** around the single best-performing (\"win-corner\") configuration, to check whether the win is robust or a knife-edge artifact of one hyperparameter choice.\n3. Derives an **analytical + microbenchmarked compute-cost comparison** between the two estimators.\n4. Documents a **methodological gap** (no short-reset-ablation baseline was ever run) and evaluates both estimators on a **real Twitter production cache trace**.\n5. Reconciles a single, corrected **memory-overhead figure** and synthesizes one final, non-hedged verdict.\n\nThis demo notebook runs the same code at a **much smaller scale** (tiny key space, few requests, few grid cells) so it completes in well under the original ~185s runtime, while keeping the exact same algorithms, formulas, and statistical machinery as the original scripts (`method.py` + `eval.py`). Config values that were shrunk for the demo are commented with their original full-scale values.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# loguru — NOT pre-installed on Colab, always install\n_pip('loguru==0.7.3')\n\n# numpy, statsmodels, matplotlib — pre-installed on Colab, install locally only (exact Colab versions)\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'statsmodels==0.14.6', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations\n\nimport json\nimport statistics\nimport time\nfrom collections import OrderedDict\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom loguru import logger\nfrom statsmodels.stats.multitest import multipletests\nimport statsmodels\n\nlogger.remove()\nlogger.add(sys.stdout, level=\"INFO\", format=\"{time:HH:mm:ss}|{level:<7}|{message}\")\n\nSTATSMODELS_VERSION = statsmodels.__version__

## Load demo data\n\n`mini_demo_data.json` is a small curated subset: (1) `full_method_out_mini`, a tiny 16-cell version of the experiment's `full_method_out.json` (2 cache ratios x 1 skew level x 4 drift scenarios x 2 seeds, vs. the original 3x3x4x3=108 cells / 36 groups), produced by running the *same* `method.py` simulator functions at a much smaller scale; and (2) `real_trace_keys_sample`, the first 2,000 requests (of 80,000) from the real Twitter `cluster026` production cache trace.

In [ ]:
GITHUB_DATA_URL = \"https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-b940ce-shadow-queue-admission-with-recency/main/round-2/evaluation-1/demo/mini_demo_data.json\"\nimport os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception:\n        pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f:\n            return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
data = load_data()\nfull_method_out_mini = data[\"full_method_out_mini\"]\nreal_trace_keys_sample = data[\"real_trace_keys_sample\"]\nprint(f\"Loaded {len(full_method_out_mini['cells'])} simulation cells, {len(real_trace_keys_sample)} real-trace requests\")

## Config\n\nAll tunable parameters, shrunk to the minimum that still produces meaningful output. Original full-scale values (from `method.py` / `eval.py`) are given in comments.

In [ ]:
# --- from method.py (simulator constants) ---\nKEY_SPACE = full_method_out_mini[\"metadata\"][\"key_space\"]          # orig: 150_000\nN_REQUESTS_MAIN = full_method_out_mini[\"metadata\"][\"n_requests_main\"]  # orig: 600_000\nRECOVERY_LOOKAHEAD_MAIN = 1200                                     # orig: 60_000\nBURST_PROB = 0.5                                                   # unchanged\nSHADOW_QUEUE_MULT = 2                                               # unchanged\nROLLING_WINDOW = 300                                                # orig: 3_000 (must be << N_REQUESTS_MAIN)\nRECOVERY_TARGET_FRAC = 0.9                                          # unchanged\nCOV_HIGH_THRESH = 1.5                                               # unchanged (default CoV tier threshold)\nCOV_LOW_THRESH = 0.5                                                # unchanged (default CoV tier threshold)\nEWMA_ALPHA = 0.3                                                    # unchanged\nMIN_OBS_FOR_CLASSIFICATION = 3                                      # unchanged\n\n# --- from eval.py STEP 1 (BH/BY-FDR bootstrap) ---\nN_RESAMPLES_BOOTSTRAP = 200                                         # orig: 1_000\n\n# --- from eval.py STEP 2 (threshold-sensitivity grid, win-corner cell only) ---\nLOWER_GRID = [0.5]                                                  # orig: [0.3, 0.5, 0.7]\nUPPER_GRID = [1.5]                                                  # orig: [1.2, 1.5, 1.8]\nWINCORNER_RATIO = full_method_out_mini[\"tuning_records\"][0][\"ratio\"]   # orig: 0.01\nWINCORNER_ALPHA = full_method_out_mini[\"tuning_records\"][0][\"alpha\"]   # orig: 1.2\nGRID_SEEDS = full_method_out_mini[\"metadata\"][\"seeds\"]              # orig: [1, 2, 3]\nGRID_SCENARIOS = [d[\"name\"] for d in full_method_out_mini[\"metadata\"][\"drift_scenarios\"]]\n\n# --- from eval.py STEP 3 (analytical + microbenchmarked compute-cost comparison) ---\nMICROBENCH_CACHE_CAPACITY = 100                                     # orig: 5_000\nMICROBENCH_N_CALLS = 2_000                                          # orig: 100_000\nMICROBENCH_N_REPEATS = 3                                             # orig: 5\n\n# --- from eval.py STEP 4 (real-trace arm) ---\nREAL_TRACE_CACHE_RATIO = 0.01                                       # unchanged

## Simulator core (from `method.py`)\n\nBoth `method.py`'s and `eval.py`'s re-simulation steps need the actual simulator classes/functions, not just the pre-computed results — the threshold grid, the microbenchmark, and the real-trace arm all call these directly. This cell is `method.py`'s W-TinyLFU admission simulator (Count-Min sketch + doorkeeper + baseline vs. proposed frequency estimators + SLRU/window cache + trace generator), copied essentially verbatim so the rest of the notebook can call it exactly as `eval.py` does via its `exp_method` module.

In [ ]:
RNG_SEED_SALT = 0x9E3779B1  # fixed odd constant for deterministic integer hashing\n\n\nclass CountMin4Bit:\n    \"\"\"Depth-4 Count-Min sketch with 4-bit saturating counters, 2 per byte.\n\n    Matches Caffeine's `FrequencySketch`: increment saturates at 15, estimate\n    is the min across rows, and `halve_all` implements the RESET_MASK trick\n    (right-shift each nibble by 1, in place, in a single pass over bytes).\n    \"\"\"\n\n    DEPTH = 4\n    _RESET_MASK = 0x77  # 0111_0111: halves both nibbles, drops each LSB\n\n    def __init__(self, num_counters: int, seed: int):\n        self.width = max(16, num_counters | 1)  # odd width reduces hash collisions across rows\n        self.table = bytearray((self.width + 1) // 2)\n        rng = np.random.default_rng(seed ^ RNG_SEED_SALT)\n        # odd multipliers for a simple deterministic multiplicative hash per row\n        self._salts = [int(x) | 1 for x in rng.integers(1, 2**31 - 1, size=self.DEPTH)]\n\n    def _pos(self, key: int, row: int) -> int:\n        return ((key ^ self._salts[row]) * self._salts[(row + 1) % self.DEPTH]) % self.width\n\n    def _get_nibble(self, pos: int) -> int:\n        b = self.table[pos >> 1]\n        return b & 0x0F if pos & 1 == 0 else (b >> 4) & 0x0F\n\n    def _set_nibble(self, pos: int, value: int) -> None:\n        idx = pos >> 1\n        b = self.table[idx]\n        if pos & 1 == 0:\n            self.table[idx] = (b & 0xF0) | value\n        else:\n            self.table[idx] = (b & 0x0F) | (value << 4)\n\n    def increment(self, key: int) -> None:\n        for row in range(self.DEPTH):\n            pos = self._pos(key, row)\n            v = self._get_nibble(pos)\n            if v < 15:\n                self._set_nibble(pos, v + 1)\n\n    def estimate(self, key: int) -> int:\n        return min(self._get_nibble(self._pos(key, row)) for row in range(self.DEPTH))\n\n    def halve_all(self) -> None:\n        table = self.table\n        mask = self._RESET_MASK\n        for i in range(len(table)):\n            table[i] = (table[i] >> 1) & mask\n\n    def memory_bytes(self) -> int:\n        return len(self.table) + self.DEPTH * 8  # counters + salts\n\n\nclass Doorkeeper:\n    \"\"\"1-bit-per-slot Bloom-style first-touch filter, cleared with the sketch.\"\"\"\n\n    def __init__(self, num_bits: int, seed: int):\n        self.num_bits = max(16, num_bits | 1)\n        self.bits = bytearray((self.num_bits + 7) // 8)\n        rng = np.random.default_rng((seed ^ 0xD1B54A35) & 0x7FFFFFFF)\n        self._salt = int(rng.integers(1, 2**31 - 1)) | 1\n\n    def _pos(self, key: int) -> int:\n        return ((key ^ self._salt) * 2654435761) % self.num_bits\n\n    def contains(self, key: int) -> bool:\n        pos = self._pos(key)\n        return bool(self.bits[pos >> 3] & (1 << (pos & 7)))\n\n    def maybe_add(self, key: int) -> bool:\n        \"\"\"Returns True iff the key was NOT already present (first touch).\"\"\"\n        pos = self._pos(key)\n        byte_idx, bit = pos >> 3, 1 << (pos & 7)\n        if self.bits[byte_idx] & bit:\n            return False\n        self.bits[byte_idx] |= bit\n        return True\n\n    def clear(self) -> None:\n        for i in range(len(self.bits)):\n            self.bits[i] = 0\n\n    def memory_bytes(self) -> int:\n        return len(self.bits) + 8\n\n\nclass GlobalResetFrequencyEstimator:\n    \"\"\"Baseline: single Count-Min sketch, reset (halved) globally on a schedule.\"\"\"\n\n    name = \"global_reset_tinylfu\"\n\n    def __init__(self, cache_capacity: int, sample_size_multiplier: int, seed: int):\n        self.sketch = CountMin4Bit(4 * cache_capacity, seed=seed)\n        self.doorkeeper = Doorkeeper(cache_capacity * 8, seed=seed + 1)\n        self.sample_size = max(1, sample_size_multiplier * cache_capacity)\n        self.size = 0\n        self.sample_size_multiplier = sample_size_multiplier\n\n    def record_access(self, key: int) -> None:\n        if not self.doorkeeper.maybe_add(key):\n            self.sketch.increment(key)\n        self.size += 1\n        if self.size >= self.sample_size:\n            self.sketch.halve_all()\n            self.doorkeeper.clear()\n            self.size = 0\n\n    def frequency(self, key: int) -> int:\n        return self.sketch.estimate(key) + (1 if self.doorkeeper.contains(key) else 0)\n\n    def memory_bytes(self) -> int:\n        return self.sketch.memory_bytes() + self.doorkeeper.memory_bytes()\n\n\nclass _LRUMeta:\n    \"\"\"Bounded LRU dict for per-key shadow metadata (read-peek vs touch-on-write).\"\"\"\n\n    def __init__(self, capacity: int):\n        self.capacity = max(1, capacity)\n        self._od: \"OrderedDict[int, tuple]\" = OrderedDict()\n\n    def peek(self, key: int):\n        return self._od.get(key)\n\n    def put_and_touch(self, key: int, value: tuple) -> None:\n        if key in self._od:\n            self._od.move_to_end(key)\n        self._od[key] = value\n        if len(self._od) > self.capacity:\n            self._od.popitem(last=False)\n\n    def __len__(self) -> int:\n        return len(self._od)\n\n    def memory_bytes(self) -> int:\n        # 5-field tuple of Python numbers + dict/OrderedDict per-entry overhead;\n        # ~120 bytes/entry is a conservative empirical estimate for this shape.\n        return len(self._od) * 120 + 200\n\n\nMIN_OBS_FOR_CLASSIFICATION = 3\n\n\nclass PerKeyDecayFrequencyEstimator:\n    \"\"\"Proposed: K tiered Count-Min sketches, each with its own halving period.\n\n    Only keys currently tracked in a bounded shadow-metadata LRU get a\n    per-key inter-arrival CoV estimate and tier assignment; a key that falls\n    out of the shadow queue reverts to the default tier on re-entry, bounding\n    memory at O(shadow_queue_capacity) regardless of the true key space.\n    \"\"\"\n\n    name = \"per_key_decay_tinylfu\"\n    TIERS = [(2, \"volatile\"), (8, \"default\"), (32, \"stable\")]\n    DEFAULT_TIER = 1\n\n    def __init__(self, cache_capacity: int, shadow_queue_capacity: int, seed: int):\n        self.tier_sketches = [\n            CountMin4Bit(4 * cache_capacity, seed=seed + 100 + t) for t in range(len(self.TIERS))\n        ]\n        self.tier_sample_size = [max(1, m * cache_capacity) for m, _ in self.TIERS]\n        self.tier_size = [0] * len(self.TIERS)\n        self.doorkeeper = Doorkeeper(cache_capacity * 8, seed=seed + 1)\n        self.shadow_meta = _LRUMeta(shadow_queue_capacity)\n        self.global_clock = 0\n        self.tier_assignment_counts = [0] * len(self.TIERS)  # diagnostics\n\n    def _classify(self, ewma_gap: float, ewma_gap_sq: float, n_obs: int) -> int:\n        if n_obs < MIN_OBS_FOR_CLASSIFICATION:\n            return self.DEFAULT_TIER\n        var = max(ewma_gap_sq - ewma_gap * ewma_gap, 0.0)\n        cov = (var**0.5) / max(ewma_gap, 1e-6)\n        if cov > COV_HIGH_THRESH:\n            return 0  # volatile / bursty\n        if cov < COV_LOW_THRESH:\n            return 2  # stable / regular\n        return 1  # default\n\n    def record_access(self, key: int) -> None:\n        self.global_clock += 1\n        meta = self.shadow_meta.peek(key)\n        if meta is None:\n            tier = self.DEFAULT_TIER\n            self.shadow_meta.put_and_touch(key, (self.global_clock, 0.0, 0.0, tier, 1))\n        else:\n            last_ts, ewma_gap, ewma_gap_sq, _prev_tier, n_obs = meta\n            gap = float(self.global_clock - last_ts)\n            if n_obs > 0:\n                ewma_gap = EWMA_ALPHA * gap + (1 - EWMA_ALPHA) * ewma_gap\n                ewma_gap_sq = EWMA_ALPHA * (gap * gap) + (1 - EWMA_ALPHA) * ewma_gap_sq\n            else:\n                ewma_gap, ewma_gap_sq = gap, gap * gap\n            n_obs += 1\n            tier = self._classify(ewma_gap, ewma_gap_sq, n_obs)\n            self.shadow_meta.put_and_touch(key, (self.global_clock, ewma_gap, ewma_gap_sq, tier, n_obs))\n\n        self.tier_assignment_counts[tier] += 1\n        if not self.doorkeeper.maybe_add(key):\n            self.tier_sketches[tier].increment(key)\n            self.tier_size[tier] += 1\n            if self.tier_size[tier] >= self.tier_sample_size[tier]:\n                self.tier_sketches[tier].halve_all()\n                self.tier_size[tier] = 0\n\n    def frequency(self, key: int) -> int:\n        meta = self.shadow_meta.peek(key)\n        tier = meta[3] if meta is not None else self.DEFAULT_TIER\n        base = self.tier_sketches[tier].estimate(key)\n        return base + (1 if self.doorkeeper.contains(key) else 0)\n\n    def memory_bytes(self) -> int:\n        return (\n            sum(s.memory_bytes() for s in self.tier_sketches)\n            + self.doorkeeper.memory_bytes()\n            + self.shadow_meta.memory_bytes()\n        )\n\n\nclass SLRUCache:\n    \"\"\"Segmented LRU: 80% protected / 20% probationary (Caffeine's default split).\"\"\"\n\n    def __init__(self, capacity: int):\n        self.capacity = max(1, capacity)\n        self.protected_capacity = max(1, int(0.8 * self.capacity))\n        self.probationary_capacity = max(1, self.capacity - self.protected_capacity)\n        self.protected: \"OrderedDict[int, None]\" = OrderedDict()\n        self.probationary: \"OrderedDict[int, None]\" = OrderedDict()\n\n    def get(self, key: int) -> bool:\n        if key in self.protected:\n            self.protected.move_to_end(key)\n            return True\n        if key in self.probationary:\n            del self.probationary[key]\n            self.protected[key] = None\n            if len(self.protected) > self.protected_capacity:\n                demoted, _ = self.protected.popitem(last=False)\n                self.probationary[demoted] = None\n                if len(self.probationary) > self.probationary_capacity:\n                    self.probationary.popitem(last=False)\n            return True\n        return False\n\n    def victim_for_admission_test(self) -> Optional[int]:\n        if self.probationary:\n            return next(iter(self.probationary))\n        return None\n\n    def admit_candidate(self, key: int) -> Optional[int]:\n        \"\"\"Admits into probationary MRU; evicts+returns probationary LRU if full.\"\"\"\n        evicted = None\n        if len(self.probationary) >= self.probationary_capacity and self.probationary:\n            evicted, _ = self.probationary.popitem(last=False)\n        self.probationary[key] = None\n        return evicted\n\n    def memory_bytes(self) -> int:\n        return (len(self.protected) + len(self.probationary)) * 56  # int key + OrderedDict entry overhead\n\n\nclass WindowTinyLFUCache:\n    \"\"\"Full W-TinyLFU: small LRU admission window + doorkeeper/sketch-gated SLRU main.\"\"\"\n\n    def __init__(self, capacity: int, estimator, window_frac: float = 0.01):\n        self.window_capacity = max(1, int(round(window_frac * capacity)))\n        self.main_capacity = max(1, capacity - self.window_capacity)\n        self.window: \"OrderedDict[int, None]\" = OrderedDict()\n        self.main = SLRUCache(self.main_capacity)\n        self.estimator = estimator\n\n    def access(self, key: int) -> bool:\n        \"\"\"Records the access with the estimator and returns True on a cache hit.\"\"\"\n        self.estimator.record_access(key)\n        if key in self.window:\n            self.window.move_to_end(key)\n            return True\n        if self.main.get(key):\n            return True\n        # miss: admit into the window; if the window overflows, its evicted\n        # LRU item competes for a main-region slot against the SLRU victim.\n        self.window[key] = None\n        if len(self.window) > self.window_capacity:\n            candidate, _ = self.window.popitem(last=False)\n            victim = self.main.victim_for_admission_test()\n            if victim is None or self.estimator.frequency(candidate) > self.estimator.frequency(victim):\n                self.main.admit_candidate(candidate)\n        return False\n\n    def memory_bytes(self) -> int:\n        return self.estimator.memory_bytes() + self.main.memory_bytes() + len(self.window) * 56\n\n\n@dataclass\nclass TraceResult:\n    keys: np.ndarray\n    drift_indices: list = field(default_factory=list)\n    burst_indices: list = field(default_factory=list)\n\n\ndef make_zipf_drift_trace(n_requests, key_space, alpha, n_drift_events, drift_magnitude, burst_prob, seed) -> TraceResult:\n    \"\"\"Zipf(alpha) popularity over `key_space` keys, with periodic hot-key\n    identity churn (drift) and occasional short bursts on a previously cold key.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    ranks = np.arange(1, key_space + 1, dtype=np.float64)\n    probs = ranks ** (-alpha)\n    probs /= probs.sum()\n    rank_to_key = np.arange(key_space, dtype=np.int64)  # identity mapping initially\n\n    n_segments = n_drift_events + 1\n    seg_len = n_requests // n_segments\n    trace = np.empty(n_requests, dtype=np.int64)\n    drift_indices: list = []\n    burst_indices: list = []\n\n    top_frac_for_drift = max(1, int(round(drift_magnitude * key_space)))\n    burst_len = 200\n\n    pos = 0\n    for seg in range(n_segments):\n        this_len = seg_len if seg < n_segments - 1 else (n_requests - pos)\n        if this_len <= 0:\n            continue\n        rank_idx = rng.choice(key_space, size=this_len, p=probs)\n        seg_keys = rank_to_key[rank_idx]\n\n        if burst_prob > 0 and rng.random() < burst_prob and this_len > burst_len + 1:\n            cold_rank = int(rng.integers(key_space // 2, key_space))\n            burst_key = int(rank_to_key[cold_rank])\n            start = int(rng.integers(0, this_len - burst_len))\n            seg_keys[start : start + burst_len] = burst_key\n            burst_indices.append(pos + start)\n\n        trace[pos : pos + this_len] = seg_keys\n        pos += this_len\n\n        if seg < n_segments - 1:\n            top_indices = np.arange(top_frac_for_drift)\n            rank_to_key[top_indices] = rng.choice(key_space, size=top_frac_for_drift, replace=False)\n            drift_indices.append(pos)\n\n    return TraceResult(keys=trace, drift_indices=drift_indices, burst_indices=burst_indices)\n\n\ndef _rolling_hit_ratio_fast(hit_bits: np.ndarray, window: int) -> np.ndarray:\n    \"\"\"O(n) rolling mean via cumulative sums.\"\"\"\n    n = len(hit_bits)\n    csum = np.cumsum(np.insert(hit_bits.astype(np.float64), 0, 0.0))\n    idx = np.arange(n)\n    lo = np.maximum(0, idx - window + 1)\n    counts = idx - lo + 1\n    return (csum[idx + 1] - csum[lo]) / counts\n\n\ndef run_trace(trace: np.ndarray, cache_capacity: int, estimator, window_admission_frac: float = 0.01) -> dict:\n    cache = WindowTinyLFUCache(cache_capacity, estimator, window_frac=window_admission_frac)\n    n = len(trace)\n    hit_bits = np.empty(n, dtype=np.uint8)\n    for i in range(n):\n        hit_bits[i] = 1 if cache.access(int(trace[i])) else 0\n    final_hit_ratio = float(hit_bits.mean())\n    rolling = _rolling_hit_ratio_fast(hit_bits, ROLLING_WINDOW)\n    return {\"final_hit_ratio\": final_hit_ratio, \"rolling_hit_ratio\": rolling, \"memory_bytes\": cache.memory_bytes()}\n\n\ndef compute_recovery_times(rolling: np.ndarray, drift_indices: list, lookahead: int = RECOVERY_LOOKAHEAD_MAIN) -> list:\n    \"\"\"For each drift point, time until rolling hit ratio climbs back to\n    RECOVERY_TARGET_FRAC of the way from the post-drift trough back to the pre-drift plateau.\n    \"\"\"\n    n = len(rolling)\n    results = []\n    for d in drift_indices:\n        pre_lo, pre_hi = max(0, d - ROLLING_WINDOW), d\n        if pre_hi <= pre_lo:\n            continue\n        plateau = float(np.mean(rolling[pre_lo:pre_hi]))\n        search_lo = d + ROLLING_WINDOW\n        post_hi = min(n, d + lookahead)\n        if post_hi <= search_lo:\n            continue\n        window = rolling[search_lo:post_hi]\n        trough = float(np.min(window))\n        target = trough + RECOVERY_TARGET_FRAC * (plateau - trough)\n        recovered_offsets = np.where(window >= target)[0]\n        if len(recovered_offsets) == 0:\n            results.append({\"drift_index\": int(d), \"recovery_time\": lookahead, \"censored\": True})\n        else:\n            results.append({\"drift_index\": int(d), \"recovery_time\": int(recovered_offsets[0]) + ROLLING_WINDOW, \"censored\": False})\n    return results\n\n\ndef _bootstrap_ci(values: list, n_resamples: int = 1000, seed: int = 0) -> dict:\n    values = [v for v in values if v is not None and not (isinstance(v, float) and np.isnan(v))]\n    if len(values) == 0:\n        return {\"mean\": None, \"ci_low\": None, \"ci_high\": None, \"n\": 0}\n    arr = np.asarray(values, dtype=np.float64)\n    rng = np.random.default_rng(seed)\n    if len(arr) == 1:\n        return {\"mean\": float(arr[0]), \"ci_low\": float(arr[0]), \"ci_high\": float(arr[0]), \"n\": 1}\n    boot_means = np.empty(n_resamples)\n    for b in range(n_resamples):\n        sample = rng.choice(arr, size=len(arr), replace=True)\n        boot_means[b] = sample.mean()\n    return {\"mean\": float(arr.mean()), \"ci_low\": float(np.percentile(boot_means, 2.5)), \"ci_high\": float(np.percentile(boot_means, 97.5)), \"n\": int(len(arr))}

## STEP 1: Benjamini-Hochberg / Benjamini-Yekutieli FDR correction\n\nGroup the loaded simulation cells by `(ratio, alpha, drift_scenario)` and compute a two-sided percentile-bootstrap p-value per group for H0 = \"no speed-up\" (proposed/baseline recovery-time ratio >= 1), then correct across all groups with `statsmodels.stats.multitest.multipletests` (BH primary, BY as a robustness check valid under arbitrary dependence — the demo dataset's 8 groups, like the original's 36, share seeds across groups, violating BH's independence assumption).

In [ ]:
def group_cells(cells: list) -> dict:\n    groups = {}\n    for c in cells:\n        key = (c[\"ratio\"], c[\"alpha\"], c[\"drift_scenario\"])\n        groups.setdefault(key, []).append(c)\n    return groups\n\n\ndef bootstrap_p_value(recov_ratios: list, n_resamples: int = N_RESAMPLES_BOOTSTRAP, seed: int = 0) -> dict:\n    \"\"\"Two-sided percentile-bootstrap p-value for H0: ratio(proposed/baseline) >= 1.\"\"\"\n    vals = [v for v in recov_ratios if v is not None and not (isinstance(v, float) and np.isnan(v))]\n    if len(vals) < 2:\n        return {\"p_value\": 1.0, \"mean\": (vals[0] if vals else None), \"n\": len(vals)}\n    arr = np.asarray(vals, dtype=np.float64)\n    rng = np.random.default_rng(seed)\n    boot_means = np.empty(n_resamples)\n    for b in range(n_resamples):\n        boot_means[b] = rng.choice(arr, size=len(arr), replace=True).mean()\n    frac_ge1 = float(np.mean(boot_means >= 1.0))\n    frac_lt1 = float(np.mean(boot_means < 1.0))\n    p = 2.0 * min(frac_ge1, frac_lt1)\n    p = min(p, 1.0)\n    # bootstrap p-values are lower-bounded by 2/n_resamples (can't observe a rarer event)\n    p = max(p, 2.0 / n_resamples)\n    return {\"p_value\": p, \"mean\": float(arr.mean()), \"n\": int(len(arr))}\n\n\ndef run_bh_fdr_analysis(cells: list) -> dict:\n    logger.info(\"STEP 1: Benjamini-Hochberg FDR correction over groups\")\n    groups = group_cells(cells)\n\n    rows = []\n    for i, (key, rows_for_group) in enumerate(sorted(groups.items())):\n        ratio, alpha, scenario = key\n        recov_ratios = []\n        for c in rows_for_group:\n            b, p = c[\"baseline\"][\"mean_recovery_time\"], c[\"proposed\"][\"mean_recovery_time\"]\n            if b and b > 0 and p is not None:\n                recov_ratios.append(p / b)\n        stat = bootstrap_p_value(recov_ratios, seed=1000 + i)\n        rows.append({\"group_id\": i, \"ratio\": ratio, \"alpha\": alpha, \"drift_scenario\": scenario,\n                     \"n_seeds\": len(recov_ratios), \"recovery_ratio_mean\": stat[\"mean\"], \"raw_p_value\": stat[\"p_value\"]})\n\n    pvals = np.array([r[\"raw_p_value\"] for r in rows])\n    reject_bh, qvals_bh, _, _ = multipletests(pvals, alpha=0.05, method=\"fdr_bh\")\n    reject_by, qvals_by, _, _ = multipletests(pvals, alpha=0.05, method=\"fdr_by\")\n\n    for r, rej_bh, q_bh, rej_by, q_by in zip(rows, reject_bh, qvals_bh, reject_by, qvals_by):\n        r[\"bh_qvalue\"] = float(q_bh)\n        r[\"bh_significant_q05\"] = bool(rej_bh)\n        r[\"by_qvalue\"] = float(q_by)\n        r[\"by_significant_q05\"] = bool(rej_by)\n\n    n_raw_sig = sum(1 for p in pvals if p < 0.05)\n    n_bh_sig = int(reject_bh.sum())\n    n_by_sig = int(reject_by.sum())\n\n    # \"win-corner\": the smallest cache ratio at the highest skew level — the\n    # config the original experiment reported the largest speed-up in.\n    win_corner_keys = {(WINCORNER_RATIO, WINCORNER_ALPHA, s) for s in GRID_SCENARIOS}\n    win_corner_rows = [r for r in rows if (r[\"ratio\"], r[\"alpha\"], r[\"drift_scenario\"]) in win_corner_keys]\n    win_corner_survive_bh = [r for r in win_corner_rows if r[\"bh_significant_q05\"]]\n    win_corner_survive_by = [r for r in win_corner_rows if r[\"by_significant_q05\"]]\n\n    logger.info(f\"raw p<0.05: {n_raw_sig}/{len(rows)} | BH q<0.05 survivors: {n_bh_sig}/{len(rows)} | \"\n                f\"BY q<0.05 survivors: {n_by_sig}/{len(rows)} | win-corner BH survivors: \"\n                f\"{len(win_corner_survive_bh)}/{len(win_corner_rows)}\")\n\n    return {\"rows\": rows, \"n_raw_significant_p05\": n_raw_sig, \"n_bh_significant_q05\": n_bh_sig,\n            \"n_by_significant_q05\": n_by_sig, \"win_corner_group_ids\": [r[\"group_id\"] for r in win_corner_rows],\n            \"win_corner_survive_bh\": [r[\"group_id\"] for r in win_corner_survive_bh],\n            \"win_corner_survive_by\": [r[\"group_id\"] for r in win_corner_survive_by]}\n\n\nbh = run_bh_fdr_analysis(full_method_out_mini[\"cells\"])

## STEP 2: CoV-threshold sensitivity grid (win-corner cell only)\n\nRe-simulates the **proposed** estimator only (the baseline is threshold-independent, so its recovery times are pulled straight from the loaded cells) across a small grid of `(COV_LOW_THRESH, COV_HIGH_THRESH)` pairs, at the win-corner `(ratio, alpha)` config, for every drift scenario and seed. This checks whether the win holds only at the exact default threshold pair or is robust nearby. The original ran this in parallel with `ProcessPoolExecutor`; the demo grid is tiny, so it runs as a plain sequential loop instead — same per-cell logic, no multiprocessing overhead needed at this scale.

In [ ]:
def _run_one_threshold_cell(lower, upper, drift_scenario, seed) -> dict:\n    \"\"\"Re-runs ONLY the proposed estimator for one (scenario, seed, lower, upper)\n    combination at the win-corner cell. Monkeypatches the module-level CoV\n    thresholds BEFORE simulating; _classify() reads these as globals on every call.\n    \"\"\"\n    global COV_LOW_THRESH, COV_HIGH_THRESH\n    COV_LOW_THRESH, COV_HIGH_THRESH = lower, upper\n\n    cache_capacity = max(10, int(WINCORNER_RATIO * KEY_SPACE))\n    tr = make_zipf_drift_trace(\n        N_REQUESTS_MAIN, KEY_SPACE, WINCORNER_ALPHA,\n        n_drift_events=drift_scenario[\"n_drift_events\"], drift_magnitude=drift_scenario[\"drift_magnitude\"],\n        burst_prob=BURST_PROB, seed=seed,\n    )\n    proposed_est = PerKeyDecayFrequencyEstimator(cache_capacity, shadow_queue_capacity=SHADOW_QUEUE_MULT * cache_capacity, seed=seed * 7 + 2)\n    proposed_res = run_trace(tr.keys, cache_capacity, proposed_est)\n    proposed_recovery = compute_recovery_times(proposed_res[\"rolling_hit_ratio\"], tr.drift_indices, lookahead=RECOVERY_LOOKAHEAD_MAIN)\n    vals = [r[\"recovery_time\"] for r in proposed_recovery]\n    mean_recovery = float(np.mean(vals)) if vals else None\n    return {\"lower\": lower, \"upper\": upper, \"drift_scenario\": drift_scenario[\"name\"], \"seed\": seed, \"proposed_mean_recovery_time\": mean_recovery}\n\n\ndef run_threshold_grid(cells: list) -> dict:\n    logger.info(\"STEP 2: threshold-sensitivity grid (win-corner cell only)\")\n    groups = group_cells(cells)\n    baseline_by_scenario_seed = {}\n    for scen in GRID_SCENARIOS:\n        for c in groups[(WINCORNER_RATIO, WINCORNER_ALPHA, scen)]:\n            baseline_by_scenario_seed[(scen, c[\"seed\"])] = c[\"baseline\"][\"mean_recovery_time\"]\n\n    drift_scenario_by_name = {d[\"name\"]: d for d in full_method_out_mini[\"metadata\"][\"drift_scenarios\"]}\n\n    t0 = time.time()\n    results = []\n    for lower in LOWER_GRID:\n        for upper in UPPER_GRID:\n            if lower >= upper:\n                continue\n            for scen in GRID_SCENARIOS:\n                for seed in GRID_SEEDS:\n                    results.append(_run_one_threshold_cell(lower, upper, drift_scenario_by_name[scen], seed))\n    logger.info(f\"Threshold grid: {len(results)} proposed-only re-simulations done in {time.time()-t0:.1f}s\")\n\n    by_combo = {}\n    for r in results:\n        b = baseline_by_scenario_seed[(r[\"drift_scenario\"], r[\"seed\"])]\n        ratio = r[\"proposed_mean_recovery_time\"] / b if (b and b > 0 and r[\"proposed_mean_recovery_time\"] is not None) else None\n        key = (r[\"lower\"], r[\"upper\"], r[\"drift_scenario\"])\n        by_combo.setdefault(key, []).append(ratio)\n\n    grid_rows = []\n    for (lower, upper, scen), ratios in sorted(by_combo.items()):\n        ci = _bootstrap_ci(ratios, n_resamples=N_RESAMPLES_BOOTSTRAP, seed=hash((lower, upper, scen)) & 0xFFFF)\n        if ci[\"mean\"] is None:\n            verdict = \"insufficient_data\"\n        elif ci[\"ci_high\"] is not None and ci[\"ci_high\"] < 1.0:\n            verdict = \"advantage_holds\"\n        elif ci[\"ci_low\"] is not None and ci[\"ci_low\"] > 1.0:\n            verdict = \"reverses\"\n        else:\n            verdict = \"advantage_narrows_or_disappears\"\n        grid_rows.append({\"lower\": lower, \"upper\": upper, \"drift_scenario\": scen, \"recovery_ratio_mean\": ci[\"mean\"],\n                           \"ci_low\": ci[\"ci_low\"], \"ci_high\": ci[\"ci_high\"], \"verdict\": verdict})\n\n    # internal consistency check: rerun at the config's default thresholds should\n    # reproduce the already-loaded proposed mean_recovery_time exactly (deterministic).\n    consistency_checks = []\n    default_lower, default_upper = 0.5, 1.5\n    if default_lower in LOWER_GRID and default_upper in UPPER_GRID:\n        for scen in GRID_SCENARIOS:\n            for c in groups[(WINCORNER_RATIO, WINCORNER_ALPHA, scen)]:\n                rerun = next(r for r in results if r[\"lower\"] == default_lower and r[\"upper\"] == default_upper and r[\"drift_scenario\"] == scen and r[\"seed\"] == c[\"seed\"])\n                orig_val, new_val = c[\"proposed\"][\"mean_recovery_time\"], rerun[\"proposed_mean_recovery_time\"]\n                delta = None if (orig_val is None or new_val is None) else abs(orig_val - new_val)\n                consistency_checks.append({\"drift_scenario\": scen, \"seed\": c[\"seed\"], \"original\": orig_val, \"rerun\": new_val, \"delta\": delta})\n    max_delta = max((c[\"delta\"] for c in consistency_checks if c[\"delta\"] is not None), default=None)\n    logger.info(f\"Internal consistency check (rerun @ default thresholds vs original): max delta = {max_delta}\")\n\n    n_holds = sum(1 for r in grid_rows if r[\"verdict\"] == \"advantage_holds\")\n    n_narrows = sum(1 for r in grid_rows if r[\"verdict\"] == \"advantage_narrows_or_disappears\")\n    n_reverses = sum(1 for r in grid_rows if r[\"verdict\"] == \"reverses\")\n    return {\"grid_rows\": grid_rows, \"consistency_check_max_abs_delta\": max_delta,\n            \"n_pairs_x_scenarios\": len(grid_rows), \"n_advantage_holds\": n_holds,\n            \"n_advantage_narrows_or_disappears\": n_narrows, \"n_reverses\": n_reverses}\n\n\ngrid = run_threshold_grid(full_method_out_mini[\"cells\"])\n# restore default thresholds for the rest of the notebook\nCOV_LOW_THRESH, COV_HIGH_THRESH = 0.5, 1.5

## STEP 3: per-request compute-cost comparison (analytical + microbenchmark)\n\nAn analytical elementary-operation count derived by reading `record_access` for each estimator, plus a wall-clock microbenchmark actually calling both estimators, reported side by side since they can diverge (branch prediction, cache locality, Python object overhead).

In [ ]:
def analytical_op_counts() -> dict:\n    \"\"\"Derived by reading GlobalResetFrequencyEstimator.record_access and\n    PerKeyDecayFrequencyEstimator.record_access line by line. Counts are per-request\n    elementary operations, amortizing periodic full-table halving over the accesses\n    between halvings.\n    \"\"\"\n    DEPTH = 4  # CountMin4Bit.DEPTH\n\n    doorkeeper_ops = 3 + 2  # _pos(3) + test/set(2)\n    sketch_increment_ops = DEPTH * (3 + 1 + 1)\n    baseline_per_request = doorkeeper_ops + sketch_increment_ops\n\n    shadow_peek_ops = 2  # dict hash + lookup (OrderedDict.get)\n    shadow_put_touch_ops = 4  # move_to_end (or not) + dict __setitem__ + len check + possible popitem\n    ewma_update_ops = 2 * 3  # 2 EWMAs (gap, gap_sq), each ~3 flops when n_obs>0\n    classify_ops = 6  # var(sub+mul), sqrt, div, 2 comparisons, tier lookup\n    tier_increment_ops = sketch_increment_ops  # identical structure, only 1 of 3 tiers touched\n    proposed_per_request = shadow_peek_ops + shadow_put_touch_ops + ewma_update_ops + classify_ops + doorkeeper_ops + tier_increment_ops\n\n    ratio = proposed_per_request / baseline_per_request\n    return {\n        \"operations\": [\n            {\"operation_type\": \"doorkeeper maybe_add\", \"baseline_count\": doorkeeper_ops, \"proposed_count\": doorkeeper_ops},\n            {\"operation_type\": \"frequency-sketch increment (DEPTH=4 hashed rows)\", \"baseline_count\": sketch_increment_ops, \"proposed_count\": tier_increment_ops},\n            {\"operation_type\": \"shadow-metadata peek\", \"baseline_count\": 0, \"proposed_count\": shadow_peek_ops},\n            {\"operation_type\": \"shadow-metadata put_and_touch\", \"baseline_count\": 0, \"proposed_count\": shadow_put_touch_ops},\n            {\"operation_type\": \"EWMA inter-arrival-gap + gap^2 update\", \"baseline_count\": 0, \"proposed_count\": ewma_update_ops},\n            {\"operation_type\": \"CoV tier reclassification\", \"baseline_count\": 0, \"proposed_count\": classify_ops},\n            {\"operation_type\": \"TOTAL per-request elementary ops (excl. amortized halving)\", \"baseline_count\": baseline_per_request, \"proposed_count\": proposed_per_request},\n        ],\n        \"proposed_over_baseline_op_ratio\": ratio,\n        \"headline\": f\"proposed does ~{ratio:.2f}x the baseline's per-request elementary-op count (excl. amortized halving)\",\n    }\n\n\ndef microbenchmark_estimators(cache_capacity=MICROBENCH_CACHE_CAPACITY, n_calls=MICROBENCH_N_CALLS, n_repeats=MICROBENCH_N_REPEATS) -> dict:\n    logger.info(f\"Microbenchmark: {n_calls} record_access calls x {n_repeats} repeats, cache_capacity={cache_capacity}\")\n    rng = np.random.default_rng(0)\n    keys = rng.integers(0, cache_capacity * 20, size=n_calls).tolist()\n\n    baseline_times, proposed_times = [], []\n    for rep in range(n_repeats):\n        est = GlobalResetFrequencyEstimator(cache_capacity, sample_size_multiplier=8, seed=rep)\n        t0 = time.perf_counter()\n        for k in keys:\n            est.record_access(k)\n        baseline_times.append(time.perf_counter() - t0)\n\n        est2 = PerKeyDecayFrequencyEstimator(cache_capacity, shadow_queue_capacity=2 * cache_capacity, seed=rep)\n        t0 = time.perf_counter()\n        for k in keys:\n            est2.record_access(k)\n        proposed_times.append(time.perf_counter() - t0)\n\n    b_mean = statistics.mean(baseline_times)\n    p_mean = statistics.mean(proposed_times)\n    return {\"n_calls\": n_calls, \"n_repeats\": n_repeats, \"baseline_seconds_mean\": b_mean, \"proposed_seconds_mean\": p_mean,\n            \"wallclock_ratio_proposed_over_baseline\": p_mean / b_mean}\n\n\ncost = {\"analytical\": analytical_op_counts(), \"microbenchmark\": microbenchmark_estimators()}\nlogger.info(cost[\"analytical\"][\"headline\"])\nlogger.info(f\"wall-clock ratio: {cost['microbenchmark']['wallclock_ratio_proposed_over_baseline']:.2f}x\")

## STEP 4: documented gap + real-trace arm\n\nFirst, an explicit check for whether a short-reset-ablation baseline exists anywhere in the loaded artifact (it does not — this is a documented gap, not something to fabricate). Second, both estimators are run once each on a sample of the **real Twitter `cluster026` production cache trace**, checking whether steady-state hit ratio parity holds within the pre-registered 1-percentage-point margin.

In [ ]:
def check_short_reset_ablation(cells: list, deviations_from_plan: list) -> dict:\n    logger.info(\"STEP 4a: checking for a short-reset-ablation baseline variant in the artifact\")\n    deviations_text = \" \".join(deviations_from_plan).lower()\n    has_short_reset = \"short\" in deviations_text and \"reset\" in deviations_text\n    has_third_variant_field = any(k not in (\"ratio\", \"alpha\", \"drift_scenario\", \"seed\", \"cache_capacity\", \"best_baseline_multiplier\", \"baseline\", \"proposed\") for k in cells[0])\n    present = has_short_reset or has_third_variant_field\n    return {\"present_in_artifact\": bool(present),\n            \"gap_statement\": None if present else (\n                \"ABSENT. The loaded cells record exactly two estimator variants — 'baseline' \"\n                \"(GlobalResetFrequencyEstimator) and 'proposed' (PerKeyDecayFrequencyEstimator). \"\n                \"No short-tuned/short-reset baseline variant aimed specifically at matching the \"\n                \"proposed estimator's drift-adaptation speed was ever run.\")}\n\n\nablation = check_short_reset_ablation(full_method_out_mini[\"cells\"], full_method_out_mini[\"metadata\"][\"deviations_from_plan\"])\nlogger.info(f\"short-reset-ablation present in artifact: {ablation['present_in_artifact']}\")\n\n\ndef run_real_trace_arm(keys_str: list, tuning_records: list) -> dict:\n    logger.info(f\"STEP 4b: real-trace arm ({len(keys_str)} Twitter cluster026 requests)\")\n    n_requests = len(keys_str)\n    distinct_keys = sorted(set(keys_str))\n    key_to_id = {k: i for i, k in enumerate(distinct_keys)}\n    trace = np.asarray([key_to_id[k] for k in keys_str], dtype=np.int64)\n    n_distinct = len(distinct_keys)\n\n    cache_capacity = max(10, int(round(REAL_TRACE_CACHE_RATIO * n_distinct)))\n    ratio_mults = [t[\"chosen_multiplier\"] for t in tuning_records if t[\"ratio\"] == REAL_TRACE_CACHE_RATIO]\n    best_multiplier = int(round(statistics.mean(ratio_mults)))\n    logger.info(f\"Real trace: n_requests={n_requests}, n_distinct_keys={n_distinct}, cache_capacity={cache_capacity}, best_multiplier={best_multiplier}\")\n\n    baseline_est = GlobalResetFrequencyEstimator(cache_capacity, best_multiplier, seed=71)\n    baseline_res = run_trace(trace, cache_capacity, baseline_est)\n    proposed_est = PerKeyDecayFrequencyEstimator(cache_capacity, shadow_queue_capacity=SHADOW_QUEUE_MULT * cache_capacity, seed=72)\n    proposed_res = run_trace(trace, cache_capacity, proposed_est)\n\n    tail_start = int(0.85 * n_requests)\n    baseline_steady = float(np.mean(baseline_res[\"rolling_hit_ratio\"][tail_start:]))\n    proposed_steady = float(np.mean(proposed_res[\"rolling_hit_ratio\"][tail_start:]))\n    delta_pp = (proposed_steady - baseline_steady) * 100.0\n    within_1pp = abs(delta_pp) <= 1.0\n\n    return {\"n_requests\": n_requests, \"n_distinct_keys\": n_distinct, \"cache_capacity\": cache_capacity,\n            \"best_multiplier_used\": best_multiplier, \"baseline_steady_state_hit_ratio\": baseline_steady,\n            \"proposed_steady_state_hit_ratio\": proposed_steady, \"steady_state_delta_percentage_points\": delta_pp,\n            \"within_preregistered_1pp_margin\": bool(within_1pp)}\n\n\nreal_trace = run_real_trace_arm(real_trace_keys_sample, full_method_out_mini[\"tuning_records\"])\nlogger.info(f\"Real-trace steady-state delta: {real_trace['steady_state_delta_percentage_points']:.3f}pp (within 1pp margin: {real_trace['within_preregistered_1pp_margin']})\")

## STEP 5: reconciled memory-overhead figure + final verdict\n\nRecomputes a single memory-overhead ratio directly from the memory-footprint table, then synthesizes all five sub-analyses (FDR survival, threshold robustness, compute cost, real-trace parity, memory overhead) into one non-hedged final verdict, following the exact same decision logic as `eval.py`.

In [ ]:
def reconcile_memory_overhead(memory_footprint_table: dict) -> dict:\n    logger.info(\"STEP 5a: recomputing the single correct memory-overhead figure\")\n    ratios = [v[\"proposed_over_baseline_ratio\"] for v in memory_footprint_table.values()]\n    return {\"min_ratio\": min(ratios), \"max_ratio\": max(ratios), \"mean_ratio\": float(np.mean(ratios)),\n            \"corrected_single_figure\": f\"{min(ratios):.2f}x-{max(ratios):.2f}x (mean {np.mean(ratios):.2f}x)\",\n            \"disconfirmation_bound_check\": {\"preregistered_bound\": \"no more than ~2x\", \"bound_exceeded\": max(ratios) > 2.0}}\n\n\ndef synthesize_final_verdict(bh, grid, cost, ablation, real_trace, memory) -> dict:\n    logger.info(\"STEP 5b: synthesizing single reconciled verdict\")\n    a_survives_bh = len(bh[\"win_corner_survive_bh\"]) > 0\n    b_robust = grid[\"n_advantage_holds\"] >= grid[\"n_pairs_x_scenarios\"] * 0.5 if grid[\"n_pairs_x_scenarios\"] else False\n    d_real_trace_corroborates = real_trace[\"within_preregistered_1pp_margin\"]\n\n    if a_survives_bh and b_robust:\n        label = \"CONFIRMED_NARROW\"\n        justification = (f\"{len(bh['win_corner_survive_bh'])}/{len(bh['win_corner_group_ids'])} win-corner groups survive BH-FDR, \"\n                          f\"and the threshold grid holds in {grid['n_advantage_holds']}/{grid['n_pairs_x_scenarios']} combinations, \"\n                          f\"but memory overhead ({memory['corrected_single_figure']}) is far above the ~2x bound.\")\n    elif not a_survives_bh:\n        label = \"DISCONFIRMED\"\n        justification = (f\"Only {bh['n_raw_significant_p05']}/{len(bh['rows'])} groups raw-significant; \"\n                          f\"{len(bh['win_corner_survive_bh'])}/{len(bh['win_corner_group_ids'])} win-corner groups survive BH-FDR. \"\n                          f\"Memory overhead {memory['corrected_single_figure']} exceeds the ~2x disconfirmation bound.\")\n    else:\n        label = \"INCONCLUSIVE_UNDERPOWERED\"\n        justification = (f\"{len(bh['win_corner_survive_bh'])}/{len(bh['win_corner_group_ids'])} win-corner groups survive BH-FDR, \"\n                          f\"but the threshold grid shows the advantage holding in only {grid['n_advantage_holds']}/{grid['n_pairs_x_scenarios']} \"\n                          f\"nearby combinations, the short-reset-ablation control was never run \"\n                          f\"(present_in_artifact={ablation['present_in_artifact']}), and the real-trace arm only supports parity \"\n                          f\"(delta={real_trace['steady_state_delta_percentage_points']:.3f}pp, within 1pp: {d_real_trace_corroborates}), \"\n                          f\"not a recovery-speed advantage. Memory overhead is {memory['corrected_single_figure']}.\")\n\n    return {\"a_survives_bh_fdr\": a_survives_bh, \"b_robust_to_threshold_choice\": bool(b_robust),\n            \"d_real_trace_corroborates_parity\": d_real_trace_corroborates, \"final_label\": label, \"justification\": justification}\n\n\nmemory = reconcile_memory_overhead(full_method_out_mini[\"memory_footprint_table\"])\nverdict = synthesize_final_verdict(bh, grid, cost, ablation, real_trace, memory)\nlogger.info(f\"FINAL VERDICT: {verdict['final_label']}\")\nlogger.info(verdict[\"justification\"])

## Results summary

In [ ]:
print(\"=\" * 70)\nprint(f\"FINAL VERDICT: {verdict['final_label']}\")\nprint(\"=\" * 70)\nprint(verdict[\"justification\"])\nprint()\n\nprint(f\"{'metric':45s} {'value':>15s}\")\nprint(\"-\" * 62)\nprint(f\"{'n groups (raw p<0.05)':45s} {bh['n_raw_significant_p05']:>6d} / {len(bh['rows']):<6d}\")\nprint(f\"{'n groups surviving BH-FDR q<0.05':45s} {bh['n_bh_significant_q05']:>6d} / {len(bh['rows']):<6d}\")\nprint(f\"{'win-corner groups surviving BH-FDR':45s} {len(bh['win_corner_survive_bh']):>6d} / {len(bh['win_corner_group_ids']):<6d}\")\nprint(f\"{'threshold-grid: advantage holds':45s} {grid['n_advantage_holds']:>6d} / {grid['n_pairs_x_scenarios']:<6d}\")\nprint(f\"{'compute cost (analytical op ratio)':45s} {cost['analytical']['proposed_over_baseline_op_ratio']:>14.2f}x\")\nprint(f\"{'compute cost (wall-clock ratio)':45s} {cost['microbenchmark']['wallclock_ratio_proposed_over_baseline']:>14.2f}x\")\nprint(f\"{'memory overhead (mean)':45s} {memory['mean_ratio']:>14.2f}x\")\nprint(f\"{'real-trace steady-state delta (pp)':45s} {real_trace['steady_state_delta_percentage_points']:>15.3f}\")\nprint(f\"{'short-reset-ablation present in artifact':45s} {str(ablation['present_in_artifact']):>15s}\")\n\n# --- plot: recovery-time ratio per group, with the FDR-corrected significance threshold ---\nfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n\nax = axes[0]\nrows_sorted = sorted(bh[\"rows\"], key=lambda r: r[\"recovery_ratio_mean\"] if r[\"recovery_ratio_mean\"] is not None else 0)\nlabels = [f\"r={r['ratio']}\\na={r['alpha']}\\n{r['drift_scenario'][:10]}\" for r in rows_sorted]\nmeans = [r[\"recovery_ratio_mean\"] for r in rows_sorted]\ncolors = [\"#2a9d8f\" if r[\"bh_significant_q05\"] else \"#e76f51\" for r in rows_sorted]\nax.bar(range(len(rows_sorted)), means, color=colors)\nax.axhline(1.0, color=\"black\", linewidth=1, linestyle=\"--\", label=\"ratio = 1 (no speed-up)\")\nax.set_xticks(range(len(rows_sorted)))\nax.set_xticklabels(labels, fontsize=7)\nax.set_ylabel(\"proposed / baseline recovery-time ratio\")\nax.set_title(\"Recovery-time ratio per group\\n(green = survives BH-FDR q<0.05)\")\nax.legend(fontsize=8)\n\nax = axes[1]\nlabels2 = [\"analytical\\n(op count)\", \"microbenchmark\\n(wall-clock)\", \"memory\\n(mean footprint)\"]\nvalues2 = [cost[\"analytical\"][\"proposed_over_baseline_op_ratio\"], cost[\"microbenchmark\"][\"wallclock_ratio_proposed_over_baseline\"], memory[\"mean_ratio\"]]\nbars = ax.bar(labels2, values2, color=[\"#457b9d\", \"#457b9d\", \"#e63946\"])\nax.axhline(1.0, color=\"black\", linewidth=1, linestyle=\"--\")\nax.axhline(2.0, color=\"gray\", linewidth=1, linestyle=\":\", label=\"pre-registered ~2x memory bound\")\nfor bar, v in zip(bars, values2):\n    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, f\"{v:.2f}x\", ha=\"center\", fontsize=9)\nax.set_ylabel(\"proposed / baseline ratio\")\nax.set_title(\"Cost of the proposed estimator\")\nax.legend(fontsize=8)\n\nplt.tight_layout()\nplt.show()